# 03 Baseline Model

This notebook implements the baseline requested:
1. Logistic Regression with `StandardScaler` and `class_weight='balanced'`
2. Benchmark metrics: AUC, recall, and business cost
3. Interpretable coefficients (including unscaled numeric coefficients)
4. OLS `statsmodels` summary (p-values and confidence intervals)

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import statsmodels.api as sm

base = Path("../datasets/processed")
df_path = base / "olist_merged_clean.csv"
df = pd.read_csv(df_path)

# Parse timestamp for temporal split
if "order_purchase_timestamp" in df.columns:
    df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"], errors="coerce")

print("Loaded:", df.shape)
print("Target late rate (%):", round(df["is_late"].mean() * 100, 2))

Loaded: (96189, 34)
Target late rate (%): 7.94


In [7]:
# Build filtered-feature set from 2.5 (core + selected engineered features)

# Core columns
core_cols = [
    "distance_km", "total_price", "total_freight", "max_weight_g", "n_items",
    "payment_installments", "same_state", "estimated_delivery_days", "product_photos_qty",
    "payment_type", "customer_state", "seller_state", "primary_category",
]

# Create only the engineered columns selected by the 2.5 cost-driven filter
# selected_extras from 2.5:
# ['hour_of_day', 'is_peak_season', 'same_city', 'product_category',
#  'seller_order_count', 'seller_state_volume', 'distance_x_weight', 'freight_x_distance']

df_model = df.copy()

df_model["hour_of_day"] = df_model["order_purchase_timestamp"].dt.hour
df_model["is_peak_season"] = df_model["order_purchase_timestamp"].dt.month.isin([4, 5, 6]).astype(int)
df_model["same_city"] = (
    df_model["customer_city"].astype(str).str.lower() ==
    df_model["seller_city"].astype(str).str.lower()
).astype(int)
df_model["product_category"] = df_model["primary_category"].fillna("unknown")

if "seller_key" not in df_model.columns:
    df_model["seller_key"] = (
        df_model["seller_zip_code_prefix"].astype(str) + "_" +
        df_model["seller_city"].astype(str) + "_" +
        df_model["seller_state"].astype(str)
    )

df_model["seller_order_count"] = df_model.groupby("seller_key")["order_id"].transform("count")
df_model["seller_state_volume"] = df_model.groupby("seller_state")["order_id"].transform("count")
df_model["distance_x_weight"] = df_model["distance_km"] * df_model["max_weight_g"]
df_model["freight_x_distance"] = df_model["total_freight"] / df_model["distance_km"].clip(lower=1)

selected_extras = [
    "hour_of_day", "is_peak_season", "same_city", "product_category",
    "seller_order_count", "seller_state_volume", "distance_x_weight", "freight_x_distance",
]

candidate_features = core_cols + selected_extras
features = [c for c in candidate_features if c in df_model.columns]

target_col = "is_late"
required_cols = features + [target_col, "order_purchase_timestamp"]
work = df_model[required_cols].dropna(subset=[target_col, "order_purchase_timestamp"]).copy()
work[target_col] = work[target_col].astype(int)

train_mask = work["order_purchase_timestamp"] < "2018-01-01"
val_mask = (work["order_purchase_timestamp"] >= "2018-01-01") & (work["order_purchase_timestamp"] < "2018-07-01")

X_train = work.loc[train_mask, features].copy()
y_train = work.loc[train_mask, target_col].copy()
X_val = work.loc[val_mask, features].copy()
y_val = work.loc[val_mask, target_col].copy()

num_features = [c for c in features if pd.api.types.is_numeric_dtype(work[c])]
cat_features = [c for c in features if c not in num_features]

print("Filtered feature set used:", features)
print("Feature count:", len(features))
print("Numeric:", num_features)
print("Categorical:", cat_features)
print("Train shape:", X_train.shape, "| Val shape:", X_val.shape)

Filtered feature set used: ['distance_km', 'total_price', 'total_freight', 'max_weight_g', 'n_items', 'payment_installments', 'same_state', 'estimated_delivery_days', 'product_photos_qty', 'payment_type', 'customer_state', 'seller_state', 'primary_category', 'hour_of_day', 'is_peak_season', 'same_city', 'product_category', 'seller_order_count', 'seller_state_volume', 'distance_x_weight', 'freight_x_distance']
Feature count: 21
Numeric: ['distance_km', 'total_price', 'total_freight', 'max_weight_g', 'n_items', 'payment_installments', 'same_state', 'estimated_delivery_days', 'product_photos_qty', 'hour_of_day', 'is_peak_season', 'same_city', 'seller_order_count', 'seller_state_volume', 'distance_x_weight', 'freight_x_distance']
Categorical: ['payment_type', 'customer_state', 'seller_state', 'primary_category', 'product_category']
Train shape: (43412, 21) | Val shape: (40270, 21)


In [11]:
# Logistic Regression baseline: StandardScaler + class_weight='balanced'
num_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_features),
        ("cat", cat_pipe, cat_features),
    ]
)

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ]
)

baseline_model.fit(X_train, y_train)
y_prob = baseline_model.predict_proba(X_val)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
business_cost = fp * 15 + fn * 155
accuracy = (tp + tn) / (tp + tn + fp + fn)

benchmark = {
    "Accuracy": accuracy,
    "AUC": roc_auc_score(y_val, y_prob),
    "Recall": recall_score(y_val, y_pred, zero_division=0),
    "Precision": precision_score(y_val, y_pred, zero_division=0),
    "F1": f1_score(y_val, y_pred, zero_division=0),
    "BusinessCost_FP15_FN155": int(business_cost),
    "FP": int(fp),
    "FN": int(fn),
    "TP": int(tp),
    "TN": int(tn),
}

pd.DataFrame([benchmark])

,Accuracy,AUC,Recall,Precision,F1,BusinessCost_FP15_FN155,FP,FN,TP,TN
0,0.742041,0.739606,0.584386,0.209208,0.308112,386120,8743,1645,2313,27569


In [9]:
# Interpretable coefficients
# Repo 1 idea: unscale numeric coefficients -> readjusted_coefs = model.coef_[0] / scaler.scale_
clf = baseline_model.named_steps["model"]
ct = baseline_model.named_steps["preprocessor"]

feature_names_out = ct.get_feature_names_out()
coef_scaled = clf.coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names_out,
    "coef_scaled": coef_scaled,
})

# Build a map of unscaled numeric coefficients only
num_scaler = ct.named_transformers_["num"].named_steps["scaler"]
num_scales = num_scaler.scale_
num_feature_names = [f"num__{c}" for c in num_features]

unscaled_map = {}
for i, f_name in enumerate(num_feature_names):
    if i < len(num_scales):
        idx = np.where(feature_names_out == f_name)[0]
        if len(idx) == 1 and num_scales[i] != 0:
            unscaled_map[f_name] = coef_scaled[idx[0]] / num_scales[i]

coef_df["coef_unscaled_numeric_only"] = coef_df["feature"].map(unscaled_map)
coef_df["abs_coef_scaled"] = coef_df["coef_scaled"].abs()

coef_df.sort_values("abs_coef_scaled", ascending=False).head(30)

,feature,coef_scaled,coef_unscaled_numeric_only,abs_coef_scaled
54,cat__seller_state_MS,-1.317467,NaN,1.317467
21,cat__customer_state_AL,1.258545,NaN,1.258545
41,cat__customer_state_RR,1.226596,NaN,1.226596
55,cat__seller_state_MT,-1.181071,NaN,1.181071
141,cat__product_category_art,-1.066680,NaN,1.066680
69,cat__primary_category_art,-1.066680,NaN,1.066680
56,cat__seller_state_PA,1.020612,NaN,1.020612
40,cat__customer_state_RO,-0.979335,NaN,0.979335
152,cat__product_category_cine_photo,0.927000,NaN,0.927000
80,cat__primary_category_cine_photo,0.927000,NaN,0.927000


In [10]:
# Repo 3 idea: OLS summary for p-values and confidence intervals
# OLS requires numeric design matrix; use numeric baseline features with median imputation.
X_train_num = X_train[num_features].copy()
X_train_num = X_train_num.fillna(X_train_num.median(numeric_only=True))

X2 = sm.add_constant(X_train_num)
model_stats = sm.OLS(y_train.values.reshape(-1, 1), X2).fit()

print(model_stats.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.017
Model:                            OLS   Adj. R-squared:                  0.017
Method:                 Least Squares   F-statistic:                     46.85
Date:                Sat, 21 Mar 2026   Prob (F-statistic):          7.89e-148
Time:                        22:18:57   Log-Likelihood:                -303.01
No. Observations:               43412   AIC:                             640.0
Df Residuals:                   43395   BIC:                             787.6
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     